# Engenharia Econômica — Aula 7: Análise de Sensibilidade e Risco

**Base:** *Engenharia Econômica - Aula 7 - Análise de Sensibilidade e Risco.pdf* — Prof. Dr. Octaviano Rojas Luiz.

Este notebook reproduz os conceitos centrais e resolve os **Exercícios 1 a 8** da aula.

## Conteúdo
- Custos fixos e variáveis
- Margem de contribuição
- Ponto de equilíbrio contábil e econômico
- Margem de segurança
- Análise de sensibilidade do VPL
- Limites de tolerância
- Ponto de indiferença entre alternativas
- Taxa de Interseção de Fisher
- Valor esperado
- Estrutura de capital, ROE e GAF

## 1. Fórmulas principais

### Margem de contribuição unitária
\[
MC = P - CV_u
\]

### Ponto de equilíbrio em unidades
\[
Q_{PE} = \frac{CF}{MC}
\]

### Quantidade para lucro-alvo
\[
Q_{alvo} = \frac{CF + Lucro_{alvo}}{MC}
\]

### Ponto de equilíbrio em receita
\[
PE_{receita} = \frac{CF}{\%MC}
\]

### VPL
\[
VPL=-I_0+\sum_{t=1}^{n}\frac{FC_t}{(1+TMA)^t}
\]

### GAF
\[
GAF = \frac{ROE}{Retorno\ Operacional}
\]

In [ ]:
import math
import pandas as pd
import matplotlib.pyplot as plt

def fator_vp_serie(taxa, n):
    return (1 - (1 + taxa)**(-n)) / taxa

def vpl_serie(investimento, fluxo_anual, taxa, n):
    return -investimento + fluxo_anual * fator_vp_serie(taxa, n)

def fator_recuperacao_capital(taxa, n):
    return taxa * (1 + taxa)**n / ((1 + taxa)**n - 1)

def caue(investimento, taxa, n):
    return investimento * fator_recuperacao_capital(taxa, n)

pd.options.display.float_format = lambda x: f"{x:,.2f}"

## 2. Exemplo da aula — Sensibilidade do Projeto Alfa

Premissas:
- investimento inicial: R$ 100.000
- receita líquida anual: R$ 40.000
- vida: 4 anos
- TMA: 10%

Vamos comparar o cenário-base com:
1. investimento inicial 20% maior;
2. receita anual 20% menor.

In [ ]:
investimento = 100_000
receita = 40_000
n = 4
tma = 0.10

cenarios = pd.DataFrame({
    "Cenário": ["Base", "Investimento +20%", "Receita -20%"],
    "Investimento": [100_000, 120_000, 100_000],
    "Receita anual": [40_000, 40_000, 32_000],
})

cenarios["VPL"] = cenarios.apply(
    lambda linha: vpl_serie(
        linha["Investimento"],
        linha["Receita anual"],
        tma,
        n
    ),
    axis=1
)

cenarios

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(cenarios["Cenário"], cenarios["VPL"])
plt.title("Sensibilidade do VPL — Projeto Alfa")
plt.ylabel("VPL (R$)")
plt.xticks(rotation=15)
plt.show()

O exemplo mostra a ideia central da aula: uma queda de 20% na receita provoca uma redução muito maior do VPL do que um aumento de 20% no investimento inicial.

## Exercício 1 — Ponto de equilíbrio e lucro-alvo

- Custos fixos: R$ 15.000/mês
- Preço: R$ 35
- Custo variável: R$ 15
- Lucro-alvo: R$ 10.000

In [ ]:
cf = 15_000
preco = 35
cv = 15
lucro_alvo = 10_000

mc = preco - cv
q_pe = cf / mc
q_alvo = (cf + lucro_alvo) / mc

print(f"Margem de contribuição = R$ {mc:,.2f}/unidade")
print(f"Ponto de equilíbrio = {q_pe:,.0f} unidades")
print(f"Quantidade para lucro de R$ 10.000 = {q_alvo:,.0f} unidades")

**Resposta:** 750 hambúrgueres para empatar e 1.250 para gerar lucro mensal de R$ 10.000.

## Exercício 2 — Ponto de equilíbrio monetário

- Custos fixos: R$ 40.000
- Margem de contribuição média: 40%

In [ ]:
cf = 40_000
mc_pct = 0.40

pe_receita = cf / mc_pct
print(f"Faturamento de equilíbrio = R$ {pe_receita:,.2f}")

**Resposta:** faturamento mensal de **R$ 100.000,00**.

## Exercício 3 — VPL base e economia anual mínima

- Investimento: R$ 80.000
- Economias anuais projetadas: R$ 30.000
- Vida: 4 anos
- TMA: 12%

In [ ]:
investimento = 80_000
economia = 30_000
n = 4
tma = 0.12

fator = fator_vp_serie(tma, n)
vpl_base = -investimento + economia * fator
economia_minima = investimento / fator
queda_toleravel = (economia - economia_minima) / economia

print(f"Fator VP = {fator:.6f}")
print(f"VPL base = R$ {vpl_base:,.2f}")
print(f"Economia anual mínima = R$ {economia_minima:,.2f}")
print(f"Queda tolerável = {queda_toleravel:.2%}")

**Resposta:** VPL ≈ **R$ 11.120,48**.  
A economia anual pode cair até aproximadamente **R$ 26.338,75** antes de o VPL zerar.

## Exercício 4 — Vida útil mínima

- Equipamento: R$ 100.000
- Fluxo anual: R$ 40.000
- TMA: 10%

Igualamos o VPL a zero para encontrar a vida de equilíbrio.

In [ ]:
investimento = 100_000
fluxo = 40_000
tma = 0.10

fator_necessario = investimento / fluxo
n_continuo = -math.log(1 - fator_necessario * tma) / math.log(1 + tma)
n_inteiro = math.ceil(n_continuo)

print(f"Prazo contínuo de equilíbrio = {n_continuo:.4f} anos")
print(f"Prazo mínimo em anos completos = {n_inteiro} anos")

In [ ]:
vidas = list(range(1, 6))
vpls = [vpl_serie(investimento, fluxo, tma, n) for n in vidas]

tabela_vida = pd.DataFrame({"Vida útil": vidas, "VPL": vpls})
tabela_vida

**Resposta:** o equilíbrio matemático ocorre em cerca de **3,02 anos**; como os fluxos são anuais, são necessários **4 anos completos**.

## Exercício 5 — Empilhadeiras e ponto de indiferença

- Elétrica: compra R$ 100.000; custo operacional R$ 20/h
- Diesel: compra R$ 60.000; custo operacional R$ 40/h
- Vida: 5 anos
- TMA: 10%

In [ ]:
tma = 0.10
vida = 5

caue_eletrica = caue(100_000, tma, vida)
caue_diesel = caue(60_000, tma, vida)

diferenca_caue = caue_eletrica - caue_diesel
economia_hora = 40 - 20
horas_indiferenca = diferenca_caue / economia_hora

print(f"CAUE Elétrica = R$ {caue_eletrica:,.2f}/ano")
print(f"CAUE Diesel   = R$ {caue_diesel:,.2f}/ano")
print(f"Ponto de indiferença = {horas_indiferenca:,.2f} horas/ano")

In [ ]:
horas = [0, 250, 500, 750, 1000]
custos = pd.DataFrame({
    "Horas/ano": horas,
    "Elétrica": [caue_eletrica + 20*h for h in horas],
    "Diesel": [caue_diesel + 40*h for h in horas],
})
custos

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(custos["Horas/ano"], custos["Elétrica"], marker="o", label="Elétrica")
plt.plot(custos["Horas/ano"], custos["Diesel"], marker="o", label="Diesel")
plt.title("Custo Total Anual — Empilhadeiras")
plt.xlabel("Horas de uso por ano")
plt.ylabel("Custo anual (R$)")
plt.legend()
plt.show()

**Resposta:** ponto de indiferença ≈ **527,60 horas/ano**.  
Abaixo disso, a Diesel tem menor custo total; acima disso, a Elétrica se torna mais econômica.

## Exercício 6 — Taxa de Interseção de Fisher

- Projeto X: -10.000 hoje; +12.500 no ano 1
- Projeto Y: -15.000 hoje; +18.000 no ano 1

In [ ]:
investimento_extra = 15_000 - 10_000
retorno_extra = 18_000 - 12_500
fisher = retorno_extra / investimento_extra - 1

tma = 0.12
vpl_x = -10_000 + 12_500 / (1 + tma)
vpl_y = -15_000 + 18_000 / (1 + tma)

print(f"Taxa de Fisher = {fisher:.2%}")
print(f"VPL X a 12% = R$ {vpl_x:,.2f}")
print(f"VPL Y a 12% = R$ {vpl_y:,.2f}")
print("Escolha a 12%:", "Projeto X" if vpl_x > vpl_y else "Projeto Y")

**Resposta:** a taxa de indiferença é **10% a.a.**.  
Com TMA de 12%, o **Projeto X** possui o maior VPL.

## Valor esperado do VPL

A aula também introduz a incorporação de probabilidades:

\[
VE = \sum VPL_i \cdot Prob_i
\]

Exemplo didático:

In [ ]:
cenarios_prob = pd.DataFrame({
    "Cenário": ["Otimista", "Base", "Pessimista"],
    "VPL": [50_000, 25_000, -20_000],
    "Probabilidade": [0.20, 0.60, 0.20],
})

cenarios_prob["Contribuição ao VE"] = (
    cenarios_prob["VPL"] * cenarios_prob["Probabilidade"]
)

valor_esperado = cenarios_prob["Contribuição ao VE"].sum()

print(f"Valor esperado do VPL = R$ {valor_esperado:,.2f}")
cenarios_prob

## Exercício 7 — Alavancagem financeira

- Investimento total: R$ 500.000
- Lucro operacional: R$ 75.000 (15%)
- 50% financiado por dívida
- Juros: 10% a.a.

In [ ]:
investimento = 500_000
lucro_operacional = 75_000
retorno_operacional = 0.15
pct_divida = 0.50
juros_divida = 0.10

divida = investimento * pct_divida
equity = investimento - divida
juros = divida * juros_divida
lucro_liquido = lucro_operacional - juros
roe = lucro_liquido / equity
gaf = roe / retorno_operacional

print(f"Dívida = R$ {divida:,.2f}")
print(f"Capital próprio = R$ {equity:,.2f}")
print(f"Juros = R$ {juros:,.2f}")
print(f"Lucro líquido = R$ {lucro_liquido:,.2f}")
print(f"ROE = {roe:.2%}")
print(f"GAF = {gaf:.2f}x")

**Resposta:** lucro líquido = **R$ 50.000**, ROE = **20%** e GAF ≈ **1,33x**.

## Exercício 8 — Ponto de equilíbrio econômico com CAUE

- Custos fixos tradicionais: R$ 80.000/ano
- Equipamento: R$ 150.000
- Vida: 5 anos
- TMA: 10%
- Preço: R$ 50/unidade
- Custo variável: R$ 20/unidade

O CAUE do equipamento é tratado como custo fixo anual.

In [ ]:
cf_tradicional = 80_000
equipamento = 150_000
vida = 5
tma = 0.10
preco = 50
cv = 20

caue_equip = caue(equipamento, tma, vida)
cf_economico = cf_tradicional + caue_equip
mc = preco - cv
pe_exato = cf_economico / mc
pe_pratico = math.ceil(pe_exato)

print(f"CAUE do equipamento = R$ {caue_equip:,.2f}/ano")
print(f"Custo fixo econômico total = R$ {cf_economico:,.2f}/ano")
print(f"PE exato = {pe_exato:,.2f} unidades/ano")
print(f"PE prático = {pe_pratico:,} unidades/ano")

**Resposta:** o ponto de equilíbrio econômico é aproximadamente **3.985,65 unidades**, portanto, na prática, **3.986 unidades por ano**.

## Resumo final

| Exercício | Tema | Resultado principal |
|---|---|---|
| 1 | Break-even | 750 unidades; 1.250 para lucro-alvo |
| 2 | PE monetário | R$ 100.000/mês |
| 3 | Limite de receita | VPL ≈ R$ 11.120,48; mínimo ≈ R$ 26.338,75/ano |
| 4 | Vida mínima | 4 anos completos |
| 5 | Ponto de indiferença | ≈ 527,60 horas/ano |
| 6 | Fisher | 10%; a 12% escolher X |
| 7 | Alavancagem | ROE = 20%; GAF ≈ 1,33x |
| 8 | PE econômico | 3.986 unidades/ano |

A lógica principal da aula é transformar risco em **limites mensuráveis**: quantas unidades, quantas horas, qual receita, qual taxa ou qual vida útil ainda preserva a viabilidade do projeto.